# Smoke test существующего pipeline в Colab

Notebook не переписывает pipeline. Он подготавливает небольшой Dataset из 64 реальных записей, после чего запускает существующие `preprocessing.pipeline` и `training.pipeline`.

Перед запуском выберите T4 GPU и загрузите `subset_5h.zip` и `manifest_5h_subset.jsonl` в `/content`.

In [1]:
%pip install -q "flash-linear-attention[cuda]==0.5.2" "transformers==5.13.1" "datasets[audio]==3.6.0" "accelerate>=1.0.0" "evaluate>=0.4.0" "jiwer>=3.0.0" "lion-pytorch>=0.2.3" "scikit-learn>=1.4.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.3/45.3 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 399.6/399.6 kB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 44.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 819.2/819.2 kB 58.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 117.8 MB/s eta 0:00:00


In [2]:
from pathlib import Path
import os
import subprocess
import sys
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA недоступна. В Colab выберите Runtime -> Change runtime type -> T4 GPU."
    )

# До принятия PR используется ваша ветка в fork.
REPO_URL = "https://github.com/SummerSchoolSPBU2026/preprocessing_and_training_dataset.git"
REPO_BRANCH = "fix/model-trainer-integration"
REPO_DIR = Path("/content/preprocessing_and_training_dataset")

if not (REPO_DIR / "training" / "pipeline.py").is_file():
    if REPO_DIR.exists():
        raise RuntimeError(
            f"{REPO_DIR} уже существует, но training/pipeline.py не найден. "
            "Удалите этот каталог через панель файлов Colab и повторите ячейку."
        )
    subprocess.run(
        [
            "git",
            "clone",
            "--branch",
            REPO_BRANCH,
            "--single-branch",
            REPO_URL,
            str(REPO_DIR),
        ],
        check=True,
    )

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

commit = subprocess.run(
    ["git", "rev-parse", "--short", "HEAD"],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()

print("Repository:", REPO_DIR)
print("Commit:", commit)
print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))

Repository: /content/preprocessing_and_training_dataset
Commit: 6270a80
PyTorch: 2.11.0+cu128
CUDA: 12.8
GPU: Tesla T4


In [3]:
from pathlib import Path

print(list(Path("/content").rglob("subset_5h.zip")))
print(list(Path("/content").rglob("manifest_5h_subset.jsonl")))

[]
[]


In [3]:
def locate_file(filename):
    candidates = [
        Path("/content") / filename,
        REPO_DIR / "training" / filename,
    ]
    for candidate in candidates:
        if candidate.is_file():
            return candidate
    raise FileNotFoundError(
        f"Не найден {filename}. Загрузите его в /content."
    )

ZIP_PATH = locate_file("subset_5h.zip")
MANIFEST_PATH = locate_file("manifest_5h_subset.jsonl")

SMOKE_EXAMPLES = 64
SMOKE_MANIFEST_PATH = Path("/content/manifest_smoke.jsonl")

with MANIFEST_PATH.open("r", encoding="utf-8") as source:
    smoke_lines = []
    for line in source:
        if line.strip():
            smoke_lines.append(line)
        if len(smoke_lines) == SMOKE_EXAMPLES:
            break

if len(smoke_lines) < SMOKE_EXAMPLES:
    raise RuntimeError(
        f"В manifest только {len(smoke_lines)} записей, ожидалось не меньше {SMOKE_EXAMPLES}."
    )

SMOKE_MANIFEST_PATH.write_text(
    "".join(smoke_lines),
    encoding="utf-8",
)

print("ZIP:", ZIP_PATH)
print("Source manifest:", MANIFEST_PATH)
print("Smoke examples:", len(smoke_lines))

FileNotFoundError: Не найден subset_5h.zip. Загрузите его в /content.

## 1. Подготовка исходного Dataset

Используется существующая функция `load_and_prepare_dataset`. Архив распаковывается целиком, но в smoke Dataset включаются только первые 64 записи.

In [ ]:
from preprocessing.prepare_dataset import load_and_prepare_dataset

raw_dataset = load_and_prepare_dataset(
    zip_path=str(ZIP_PATH),
    metadata_path=str(SMOKE_MANIFEST_PATH),
    audio_root="subset_5h",
    output_dir="prepared_data",
)

print(raw_dataset)

## 2. Существующий preprocessing pipeline

Команда ниже запускает `preprocessing/pipeline.py` без копирования его логики в notebook.

In [ ]:
subprocess.run(
    [sys.executable, "-m", "preprocessing.pipeline"],
    cwd=REPO_DIR,
    check=True,
)

prepared_path = (
    REPO_DIR
    / "preprocessed_dataset"
    / "whisper_prepocessed_dataset.arrow"
)
if not prepared_path.is_dir():
    raise RuntimeError(f"Pipeline не создал {prepared_path}")

print("Preprocessing pipeline: OK")

## 3. Существующий training pipeline

`training/pipeline.py` сам создаёт feature extractor, tokenizer, WhisperKDA, Lion, проверяет cache и запускает Trainer. Сейчас в `training/trainer.py` установлено `max_steps=2`, поэтому это smoke test, а не полное обучение.

Первый запуск KDA на T4 может несколько минут компилировать Triton kernels.

In [ ]:
subprocess.run(
    [sys.executable, "-m", "training.pipeline"],
    cwd=REPO_DIR,
    check=True,
)

print("Existing training pipeline smoke test: OK")